In [1]:
import os
import re
import ast
import json
import time
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin, urlparse, unquote, quote

import spacy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from gensim.models.fasttext import load_facebook_vectors

In [ ]:
CATEGORY_URLS = [
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pageuntil=Baklava+with+Pistachio+Nuts%0ABaklava+with+Pistachio+Nuts#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Baklava+with+Pistachio+Nuts%0ABaklava+with+Pistachio+Nuts#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Bouch%C3%A9e+%C3%A0+la+Reine%0ABouch%C3%A9e+%C3%A0+la+Reine#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Cheese+Omelette%0ACheese+Omelette#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Chunky+Cran+Apple+Bran+Muffins%0AChunky+Cran+Apple+Bran+Muffins#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Dal+Makhani+%28Black+Gram+with+Cream%29%0ADal+Makhani+%28Black+Gram+with+Cream%29#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Fattah+%28Egyptian+Layered+Bread+Dish%29%0AFattah+%28Egyptian+Layered+Bread+Dish%29#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Ginger-Glazed+Grilled+Peaches%0AGinger-Glazed+Grilled+Peaches#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Homemade+Tahini%0AHomemade+Tahini#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Key+Lime+Meringue+Pie%0AKey+Lime+Meringue+Pie#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Maff%C3%A9+%28West+African+Peanut+Stew%29+I%0AMaff%C3%A9+%28West+African+Peanut+Stew%29+I#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Moqueca+de+Peixe+%28Brazilian+Seafood+Stew%29%0AMoqueca+de+Peixe+%28Brazilian+Seafood+Stew%29#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Orange-Glazed+Sweet+Potatoes%0AOrange-Glazed+Sweet+Potatoes#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Popcorn%0APopcorn#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Rice+Cooked+in+Tomato+Meat+Sauce+%28Ross+il-Forn%29%0ARice+Cooked+in+Tomato+Meat+Sauce+%28Ross+il-Forn%29#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Semolina+Burger%0ASemolina+Burger#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Spicy+Corn+Bread%0ASpicy+Corn+Bread#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Tahini+Goddess+Dressing%0ATahini+Goddess+Dressing#mw-pages",
    "https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Upma+%28Indian+Semolina+Porridge%29%0AUpma+%28Indian+Semolina+Porridge%29#mw-pages",
]

BASE = "https://en.wikibooks.org"
API = "https://en.wikibooks.org/w/api.php"
HEADERS = {
    "User-Agent": "RecipeIngredientProject/1.0",
    "Accept-Language": "en",
}

RAW_CSV = "recipes_raw_large.csv"
PREPROCESSED_CSV = "final_recipes_dataset_large.csv"

OLLAMA_BASE = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:3b"

FT_BIN = "cc.en.300.bin"
MODEL_PT = "recipe_fasttext_mlp.pt"
LABELS_JS = "recipe_fasttext_mlp_labels.json"

In [3]:
def extract_article_links_from_category_html(url: str, session: requests.Session) -> list[str]:
    r = session.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    mw_pages = soup.select_one("#mw-pages")
    if not mw_pages:
        return []

    links = []
    for a in mw_pages.select("div.mw-category-group a[href]"):
        href = a["href"]
        if href.startswith("/wiki/") and not href.startswith("/wiki/Category:"):
            links.append(urljoin(BASE, href))

    seen = set()
    out = []
    for x in links:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


all_links = []
with requests.Session() as s:
    for url in CATEGORY_URLS:
        time.sleep(2)
        batch = extract_article_links_from_category_html(url, s)
        print(f"{len(batch):4d} | {url}")
        all_links.extend(batch)

all_links = list(dict.fromkeys(all_links))

 200 | https://en.wikibooks.org/w/index.php?title=Category:Recipes&pageuntil=Baklava+with+Pistachio+Nuts%0ABaklava+with+Pistachio+Nuts#mw-pages
 200 | https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Baklava+with+Pistachio+Nuts%0ABaklava+with+Pistachio+Nuts#mw-pages
 200 | https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Bouch%C3%A9e+%C3%A0+la+Reine%0ABouch%C3%A9e+%C3%A0+la+Reine#mw-pages
 200 | https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Cheese+Omelette%0ACheese+Omelette#mw-pages
 200 | https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Chunky+Cran+Apple+Bran+Muffins%0AChunky+Cran+Apple+Bran+Muffins#mw-pages
 200 | https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Dal+Makhani+%28Black+Gram+with+Cream%29%0ADal+Makhani+%28Black+Gram+with+Cream%29#mw-pages
 200 | https://en.wikibooks.org/w/index.php?title=Category:Recipes&pagefrom=Fattah+%28Egyptian+Layered+Bread+Dish%29%0AFattah+%28Egyp

In [4]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, unquote, quote

API = "https://en.wikibooks.org/w/api.php"
BASE = "https://en.wikibooks.org"

HEADERS = {
    "User-Agent": "JanekOpalaRecipeScraper/1.0",
    "Accept-Language": "en",
}

_HTML_CACHE = {}
_API_CACHE = {}

def _clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"\[\s*\d+\s*\]", "", s or "")).strip()

def _url_to_page_title(recipe_url: str) -> str:
    p = urlparse(recipe_url).path
    if not p.startswith("/wiki/"):
        raise ValueError(recipe_url)
    return unquote(p[6:]).replace("_", " ")

def get_soup(url: str, session: requests.Session, sleep_s: float = 0.8) -> BeautifulSoup:
    if url in _HTML_CACHE:
        return _HTML_CACHE[url]

    time.sleep(sleep_s)
    r = session.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    _HTML_CACHE[url] = soup
    return soup

def _parse_api(recipe_url: str, session: requests.Session) -> BeautifulSoup:
    if recipe_url in _API_CACHE:
        return _API_CACHE[recipe_url]

    page_title = _url_to_page_title(recipe_url)
    last_exc = None

    for attempt in range(5):
        try:
            time.sleep(1.5 + attempt)
            r = session.get(
                API,
                params={
                    "action": "parse",
                    "page": page_title,
                    "prop": "text",
                    "format": "json",
                    "formatversion": "2",
                    "redirects": "1",
                },
                headers=HEADERS,
                timeout=30,
            )
            r.raise_for_status()

            if "application/json" not in r.headers.get("Content-Type", ""):
                raise ValueError(f"Non-JSON response: {r.headers.get('Content-Type')}")

            j = r.json()
            if "parse" not in j or "text" not in j["parse"]:
                raise ValueError(f"Malformed API response for {page_title}")

            soup = BeautifulSoup(j["parse"]["text"], "html.parser")
            _API_CACHE[recipe_url] = soup
            return soup

        except Exception as e:
            last_exc = e
            time.sleep(3 * (attempt + 1))

    raise last_exc

def _get_recipe_soup(recipe_url: str, session: requests.Session) -> BeautifulSoup:
    return get_soup(recipe_url, session, sleep_s=0.8)

def extract_title(recipe_url: str, session: requests.Session | None = None) -> str:
    own = session is None
    s = session or requests.Session()
    try:
        soup = _get_recipe_soup(recipe_url, s)
        h1 = soup.select_one("h1#firstHeading")
        if h1:
            return _clean_text(h1.get_text(" ", strip=True))
        return _clean_text(_url_to_page_title(recipe_url))
    finally:
        if own:
            s.close()

def extract_photo(recipe_url: str, out_dir: str = "recipe_images", session: requests.Session | None = None, width: int = 1000) -> str | None:
    own = session is None
    s = session or requests.Session()
    try:
        soup = _get_recipe_soup(recipe_url, s)

        a = soup.select_one("table.infobox td.infobox-image a[href^='/wiki/File:'], td.infobox-image a[href^='/wiki/File:']")
        if not a:
            return None

        fn = unquote(a["href"].split("/wiki/", 1)[-1]).split("File:", 1)[-1].strip().replace(" ", "_")
        return f"{BASE}/wiki/Special:FilePath/{quote(fn)}?width={width}"
    finally:
        if own:
            s.close()

def extract_ingredients(recipe_url: str, session: requests.Session | None = None) -> list[str]:
    own = session is None
    s = session or requests.Session()
    try:
        soup = _get_recipe_soup(recipe_url, s)

        h = soup.find(id="Ingredients")
        if h and h.name not in ("h2", "h3", "h4"):
            h = h.find_parent(["h2", "h3", "h4"])

        if not h:
            h = soup.find(
                lambda t: t.name in ("h2", "h3", "h4")
                and _clean_text(t.get_text()).lower().startswith("ingredients")
            )

        out = []
        if h:
            for x in h.find_all_next():
                if x.name in ("h2", "h3", "h4") and x is not h:
                    break

                if x.name in ("ul", "ol"):
                    vals = [
                        _clean_text(li.get_text(" ", strip=True))
                        for li in x.select("li")
                    ]
                    out.extend([v for v in vals if v])

                if x.name == "table":
                    for tr in x.select("tr"):
                        tds = tr.select("td")
                        if not tds:
                            continue
                        name = _clean_text(tds[0].get_text(" ", strip=True))
                        if name:
                            out.append(name)

        if not out:
            try:
                soup_api = _parse_api(recipe_url, s)
                h = soup_api.find(id="Ingredients")
                if h and h.name not in ("h2", "h3", "h4"):
                    h = h.find_parent(["h2", "h3", "h4"])

                if not h:
                    h = soup_api.find(
                        lambda t: t.name in ("h2", "h3", "h4")
                        and _clean_text(t.get_text()).lower().startswith("ingredients")
                    )

                if h:
                    for x in h.find_all_next():
                        if x.name in ("h2", "h3", "h4") and x is not h:
                            break

                        if x.name in ("ul", "ol"):
                            vals = [
                                _clean_text(li.get_text(" ", strip=True))
                                for li in x.select("li")
                            ]
                            out.extend([v for v in vals if v])

                        if x.name == "table":
                            for tr in x.select("tr"):
                                tds = tr.select("td")
                                if not tds:
                                    continue
                                name = _clean_text(tds[0].get_text(" ", strip=True))
                                if name:
                                    out.append(name)
            except Exception:
                pass

        out = list(dict.fromkeys([x for x in out if x]))
        return out

    finally:
        if own:
            s.close()

In [7]:
recipes_df = pd.read_csv(RAW_CSV)

In [8]:
recipes_df["IngredientsKey"] = recipes_df["Ingredients"].apply(
    lambda x: tuple(sorted(set(x))) if isinstance(x, list) else x
)
recipes_df = recipes_df.drop_duplicates(subset=["Title", "IngredientsKey"]).drop(columns=["IngredientsKey"]).reset_index(drop=True)

print(recipes_df.shape)
recipes_df.head()

(3356, 3)


,Title,Ingredients,Photo
0,Cookbook:'Out of Salad Dressing' Salad Dressing,"['1 ½ lemons , juiced', '1 cup freshly grated ...",NaN
1,Cookbook:1-2-3-4 Cake,"['1 cup (240 g / 8.5 oz ) butter', '1 cup (240...",NaN
2,Cookbook:20-Minute Beef Stroganoff,['8 oz (230 g) wide egg noodles (figures are f...,NaN
3,Cookbook:40 Cloves in a Roast Chicken,"['1 ea . (3–4 pounds ) broiler/fryer chicken',...",NaN
4,Cookbook:A Nice Cup of Tea,"['32 fl oz (1 L) hot water', '3–5 measures of ...",https://en.wikibooks.org/wiki/Special:FilePath...


In [9]:
recipes_df.to_csv(RAW_CSV, index=False)

In [10]:
def ollama_generate(prompt: str, timeout: int = 60, max_retries: int = 5) -> str:
    url = f"{OLLAMA_BASE}/api/generate"
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 32,
            "temperature": 0.0,
        },
    }

    for attempt in range(max_retries):
        try:
            r = requests.post(url, json=payload, timeout=timeout)
            r.raise_for_status()
            return r.json().get("response", "").strip()
        except (
            requests.exceptions.ReadTimeout,
            requests.exceptions.ConnectTimeout,
            requests.exceptions.ConnectionError,
            requests.exceptions.HTTPError,
        ):
            time.sleep(2 ** attempt)

    return ""

_ing_cache = {}
_title_cache = {}

def extract_main_ingredient_one(ingredient: str) -> str:
    s = str(ingredient).strip()
    if not s:
        return ""
    if s in _ing_cache:
        return _ing_cache[s]

    prompt = (
        "Extract the main ingredient name from the string below.\n"
        "Return ONLY the ingredient name, no units, no quantities, no explanation.\n"
        "Examples:\n"
        "2 cups of flour -> flour\n"
        "1 tsp vanilla extract -> vanilla extract\n"
        "3 eggs -> eggs\n"
        f"String: {s}\n"
        "Answer:"
    )

    ans = ollama_generate(prompt, timeout=60, max_retries=5).strip(" \"'`.\n\t")
    if not ans:
        tmp = re.sub(r"^[\d\s\/\.\(\)\-\–¼½¾⅓⅔]+", "", s)
        tmp = re.sub(
            r"\b(cup|cups|tbsp|tsp|teaspoon|teaspoons|tablespoon|tablespoons|oz|g|kg|mg|ml|l|lb|lbs|pound|pinch|pinches|clove|cloves|slice|slices)\b",
            "",
            tmp,
            flags=re.I,
        )
        tmp = re.sub(r"\s+", " ", tmp).strip(" ,.;:-")
        ans = tmp

    _ing_cache[s] = ans
    return ans

def strip_prefix(title: str) -> str:
    s = str(title).strip()
    s = re.sub(r"^(Cookbook|User)\s*:\s*", "", s, flags=re.I)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_title_one(title: str) -> str:
    s0 = strip_prefix(title)
    if not s0:
        return ""
    if s0 in _title_cache:
        return _title_cache[s0]

    prompt = (
        "Normalize the recipe title.\n"
        "Return ONLY the generalized dish name.\n"
        "Remove origin/nationality adjectives and parenthetical translations.\n"
        "Remove author/user names, file paths, and extra descriptors.\n"
        "If you cannot confidently generalize, return the cleaned original title.\n"
        "Examples:\n"
        "Zupa Ogórkowa (Polish Cucumber Soup) -> Cucumber soup\n"
        "Aadun (Nigerian Corn Flour with Palm Oil) -> Corn flour with palm oil\n"
        "Æbleskiver (Danish Spherical Pancakes) -> Spherical pancakes\n"
        f"Title: {s0}\n"
        "Answer:"
    )

    ans = ollama_generate(prompt, timeout=60, max_retries=5).strip(" \"'`.\n\t")
    if not ans:
        ans = re.sub(r"\s*\([^)]*\)\s*", " ", s0).strip()
        ans = re.sub(r"\s+", " ", ans).strip()

    _title_cache[s0] = ans
    return ans

In [ ]:
def parse_ingredients_cell(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

recipes_df = pd.read_csv(RAW_CSV)
recipes_df["Ingredients"] = recipes_df["Ingredients"].apply(parse_ingredients_cell)

recipes_df["MainIngredients"] = [
    [extract_main_ingredient_one(ing) for ing in tqdm(ings, leave=False)]
    for ings in tqdm(recipes_df["Ingredients"], desc="Main ingredients")
]

recipes_df["NormTitle"] = [
    normalize_title_one(t) for t in tqdm(recipes_df["Title"], desc="Normalize titles")
]

recipes_df[["Title", "NormTitle", "MainIngredients"]].head()

Normalize titles: 100%|██████████| 3356/3356 [1:02:58<00:00,  1.13s/it]


,Title,NormTitle,MainIngredients
0,Cookbook:'Out of Salad Dressing' Salad Dressing,Salad dressing,"[lemons, Parmesan, garlic salt, mayonnaise, milk]"
1,Cookbook:1-2-3-4 Cake,Cake,"[butter, milk, vanilla extract, sugar, flour, ..."
2,Cookbook:20-Minute Beef Stroganoff,Beef Stroganoff,"[egg noodles, olive oil, mushrooms, onion, bee..."
3,Cookbook:40 Cloves in a Roast Chicken,Roast chicken,"[chicken, Country Roast Chicken Seasoning, gar..."
4,Cookbook:A Nice Cup of Tea,Nice cup of tea,"[water, tea, sugar, milk]"


In [12]:
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def clean_phrase(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def lemma_phrase_spacy(phrase: str) -> str:
    p = clean_phrase(phrase)
    if not p:
        return ""

    doc = nlp(p)
    toks = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue
        if t.like_num:
            continue
        if t.pos_ in {"DET"}:
            continue

        lem = t.lemma_.lower().strip()
        if lem in {"-pron-", ""}:
            lem = t.text.lower()
        toks.append(lem)

    return " ".join(toks).strip()

canonical = {}

def normalize_list_with_canonical(ingredients_list):
    out = []
    for ing in ingredients_list:
        raw = clean_phrase(ing)
        if not raw:
            out.append("")
            continue

        lem = lemma_phrase_spacy(raw)
        if not lem:
            lem = raw

        if lem in canonical:
            out.append(canonical[lem])
        else:
            canonical[lem] = lem
            out.append(lem)
    return out

recipes_df["NormalizedMainIngredients"] = recipes_df["MainIngredients"].apply(normalize_list_with_canonical)

df2 = recipes_df[["NormTitle", "NormalizedMainIngredients"]].copy()
df2["NormalizedMainIngredients"] = df2["NormalizedMainIngredients"].apply(
    lambda x: x if isinstance(x, list) else []
)

dummies = (
    df2["NormalizedMainIngredients"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)
dummies = dummies[dummies.ne("")]

X_ing = pd.crosstab(dummies.index, dummies)

df_final = (
    df2.drop(columns=["NormalizedMainIngredients"])
    .join(X_ing)
    .fillna(0)
    .astype({c: "int8" for c in X_ing.columns})
)

df_final.to_csv(PREPROCESSED_CSV, index=False)
print(df_final.shape)
df_final.head()

(3356, 2568)


,NormTitle,abacha,acelga,achara,achiote annatto,acini di pepe,ackawi cheese,acorn,acorn squash,additional ingredient as desire,...,yuca root,yukon gold,yukon gold potato,z ug,za atar,zheera,ziti,zucchini,zucchinis,zuckerhut
0,Salad dressing,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Cake,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Beef Stroganoff,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Roast chicken,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Nice cup of tea,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Filtering out some ingredients

In [13]:
min_freq_ratio = 0.01
min_count = max(1, int(np.ceil(len(df_final) * min_freq_ratio)))

label_candidate_cols = [c for c in df_final.columns if c != "NormTitle"]
label_sums = df_final[label_candidate_cols].sum(axis=0)

kept_label_cols = label_sums[label_sums >= min_count].index.tolist()
removed_label_cols = label_sums[label_sums < min_count].index.tolist()

df_final = df_final[["NormTitle"] + kept_label_cols].copy()

df_final.to_csv(PREPROCESSED_CSV, index=False)

In [14]:
_tok_re = re.compile(r"[a-zA-Z][a-zA-Z0-9_'-]*")

_STOP = {
    "a","an","the","and","or","but","of","to","in","on","for","with","without","from","by","at","as",
    "is","are","was","were","be","been","being","into","over","under","up","down","off","out","about",
    "this","that","these","those","it","its","your","my","our","their"
}

def tokenize(text: str):
    toks = _tok_re.findall((text or "").lower())
    toks = [t for t in toks if len(t) > 1 and t not in _STOP]
    if len(toks) >= 2:
        toks += [f"{toks[i]}_{toks[i+1]}" for i in range(len(toks) - 1)]
    return toks

ft = load_facebook_vectors(FT_BIN)
embedding_dim = ft.vector_size
print("embedding_dim =", embedding_dim)

def title_vector(title: str) -> np.ndarray:
    toks = tokenize(title)
    if not toks:
        return np.zeros(embedding_dim, dtype=np.float32)
    vecs = np.stack([ft.get_vector(t) for t in toks]).astype(np.float32)
    return vecs.mean(axis=0)

embedding_dim = 300


In [15]:
data = pd.read_csv(PREPROCESSED_CSV)
data["NormTitle"] = data["NormTitle"].fillna("")

candidate_cols = [c for c in data.columns if c != "NormTitle"]
label_cols = []

for c in candidate_cols:
    s = pd.to_numeric(data[c], errors="coerce")
    if s.isna().mean() > 0.01:
        continue
    vals = set(s.dropna().unique().tolist())
    if vals.issubset({0, 1}):
        label_cols.append(c)

data[label_cols] = data[label_cols].astype(np.float32)

train_df, temp_df = train_test_split(data, test_size=0.2, random_state=123, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=123, shuffle=True)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("n_labels =", len(label_cols))
print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

class FastTextDataset(Dataset):
    def __init__(self, df, label_cols):
        self.texts = df["NormTitle"].tolist()
        self.Y = df[label_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        x = title_vector(self.texts[i])
        y = self.Y[i]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

train_ds = FastTextDataset(train_df, label_cols)
val_ds = FastTextDataset(val_df, label_cols)
test_ds = FastTextDataset(test_df, label_cols)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

n_labels = 80
train: (2684, 131) val: (336, 131) test: (336, 131)


In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class MLPBN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

hidden_dim = 256
dropout = 0.4
threshold = 0.2

model = MLPBN(embedding_dim, hidden_dim, len(label_cols), dropout=dropout).to(device)
model

MLPBN(
  (net): Sequential(
    (0): Linear(in_features=300, out_features=256, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=256, out_features=80, bias=True)
  )
)

In [17]:
train_Y = train_df[label_cols].values.astype(np.float32)
pos = train_Y.sum(axis=0)
neg = train_Y.shape[0] - pos

pos_weight = torch.tensor(
    np.sqrt((neg + 1e-6) / (pos + 1e-6)),
    dtype=torch.float32
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

@torch.no_grad()
def jaccard_from_logits(logits, y_true, threshold=0.2):
    probs = torch.sigmoid(logits)
    y_pred = (probs >= threshold).float()
    inter = (y_pred * y_true).sum(dim=1)
    union = ((y_pred + y_true) > 0).float().sum(dim=1)
    return (inter / (union + 1e-9)).mean().item()

@torch.no_grad()
def micro_f1_from_logits(logits, y_true, threshold=0.2):
    probs = torch.sigmoid(logits)
    y_pred = (probs >= threshold).float()
    tp = (y_pred * y_true).sum().item()
    fp = (y_pred * (1 - y_true)).sum().item()
    fn = ((1 - y_pred) * y_true).sum().item()

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return f1, precision, recall

@torch.no_grad()
def precision_recall_at_k(logits, y_true, k=10):
    topk = torch.topk(logits, k=k, dim=1).indices
    y_true_bool = y_true.bool()

    p_list, r_list = [], []
    for i in range(logits.size(0)):
        pred_idx = topk[i]
        hit = y_true_bool[i, pred_idx].sum().item()
        true_cnt = y_true_bool[i].sum().item()

        p_list.append(hit / k)
        r_list.append(hit / (true_cnt + 1e-9))

    return float(np.mean(p_list)), float(np.mean(r_list))

In [18]:
def run_epoch(loader, train, model, criterion, optimizer, device):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_logits, all_y = [], []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for xb, yb in tqdm(loader, disable=not train):
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu())
            all_y.append(yb.detach().cpu())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, torch.cat(all_logits, dim=0), torch.cat(all_y, dim=0)


num_epochs = 20
best_val_j = -1.0
best_state = None

for epoch in range(num_epochs):
    tr_loss, _, _ = run_epoch(train_loader, True, model, criterion, optimizer, device)
    va_loss, va_logits, va_y = run_epoch(val_loader, False, model, criterion, optimizer, device)

    va_f1, va_p, va_r = micro_f1_from_logits(va_logits, va_y, threshold=threshold)
    va_j = jaccard_from_logits(va_logits, va_y, threshold=threshold)
    p10, r10 = precision_recall_at_k(va_logits, va_y, k=10)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"train loss {tr_loss:.4f} | val loss {va_loss:.4f} | "
        f"micro-F1 {va_f1:.4f} (P {va_p:.4f} R {va_r:.4f}) | "
        f"Jacc {va_j:.4f} | P@10 {p10:.4f} | R@10 {r10:.4f}"
    )

    if va_j > best_val_j:
        best_val_j = va_j
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

100%|██████████| 42/42 [00:00<00:00, 60.75it/s]


Epoch 01/20 | train loss 0.7735 | val loss 0.7125 | micro-F1 0.0404 (P 0.0206 R 1.0000) | Jacc 0.0206 | P@10 0.0726 | R@10 0.3232


100%|██████████| 42/42 [00:00<00:00, 81.20it/s] 


Epoch 02/20 | train loss 0.6259 | val loss 0.4989 | micro-F1 0.0417 (P 0.0213 R 0.9964) | Jacc 0.0212 | P@10 0.0917 | R@10 0.4118


100%|██████████| 42/42 [00:00<00:00, 80.70it/s] 


...

Epoch 18/20 | train loss 0.1828 | val loss 0.3458 | micro-F1 0.2221 (P 0.1400 R 0.5361) | Jacc 0.1188 | P@10 0.1018 | R@10 0.4580


100%|██████████| 42/42 [00:00<00:00, 61.47it/s]


Epoch 19/20 | train loss 0.1814 | val loss 0.3445 | micro-F1 0.2130 (P 0.1327 R 0.5397) | Jacc 0.1126 | P@10 0.0985 | R@10 0.4508


100%|██████████| 42/42 [00:00<00:00, 61.52it/s]


Epoch 20/20 | train loss 0.1759 | val loss 0.3508 | micro-F1 0.2197 (P 0.1384 R 0.5325) | Jacc 0.1134 | P@10 0.0994 | R@10 0.4615


In [19]:
model.load_state_dict(best_state)

te_loss, te_logits, te_y = run_epoch(test_loader, False, model, criterion, optimizer, device)
te_f1, te_p, te_r = micro_f1_from_logits(te_logits, te_y, threshold=threshold)
te_j = jaccard_from_logits(te_logits, te_y, threshold=threshold)
p10, r10 = precision_recall_at_k(te_logits, te_y, k=10)

print("=== TEST ===")
print(
    f"loss {te_loss:.4f} | "
    f"micro-F1 {te_f1:.4f} (P {te_p:.4f} R {te_r:.4f}) | "
    f"Jacc {te_j:.4f} | P@10 {p10:.4f} | R@10 {r10:.4f}"
)

=== TEST ===
loss 0.3440 | micro-F1 0.2015 (P 0.1259 R 0.5049) | Jacc 0.1037 | P@10 0.0899 | R@10 0.4400


In [20]:
@torch.no_grad()
def predict_title(title, topk=10):
    model.eval()
    x = torch.tensor(title_vector(title), dtype=torch.float32).unsqueeze(0).to(device)
    logits = model(x).cpu().squeeze(0)
    idx = torch.topk(logits, k=topk).indices.numpy().tolist()
    labels = [label_cols[i] for i in idx]
    scores = torch.sigmoid(logits[idx]).numpy().tolist()
    return list(zip(labels, scores))

predict_title("pizza", topk=10)

[('mozzarella cheese', 0.9413390159606934),
 ('tomato sauce', 0.9188092350959778),
 ('oregano', 0.8551392555236816),
 ('chili powder', 0.35100528597831726),
 ('ground beef', 0.22734592854976654),
 ('bake powder', 0.09218785166740417),
 ('tomato paste', 0.08173508942127228),
 ('cornstarch', 0.05054573714733124),
 ('vanilla extract', 0.0479111522436142),
 ('shortening', 0.04450305551290512)]

In [21]:
artifact = {
    "state_dict": model.state_dict(),
    "embedding_dim": int(embedding_dim),
    "hidden_dim": int(hidden_dim),
    "label_cols": label_cols,
    "threshold": float(threshold),
}

torch.save(artifact, MODEL_PT)

with open(LABELS_JS, "w", encoding="utf-8") as f:
    json.dump(label_cols, f, ensure_ascii=False, indent=2)

print("Saved:", MODEL_PT, "and", LABELS_JS)

Saved: recipe_fasttext_mlp.pt and recipe_fasttext_mlp_labels.json
